# 03 — Summary Statistics Design

Investigate which summary statistics are informative for which parameters.

**Key question:** beta and rho can both suppress the epidemic. Can infected-only
summaries separate them? How much do rewiring and degree summaries help?

**Experiments:**
1. Run rejection ABC with 6 different summary subsets
2. Compare posterior widths and correlations
3. Show how rewiring/degree stats break the beta-rho degeneracy

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import corner
import time

from data_loader import load_all
from summary_stats import (compute_observed_summaries, compute_summaries, SUMMARY_NAMES,
                           IDX_INFECTED, IDX_REWIRING, IDX_DEGREE, IDX_ALL)
from abc_rejection import run_rejection_abc, accept_quantile
from simulator import simulate_fast

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load data
inf_ts, rew_ts, deg_hist = load_all()
s_obs, s_per_rep = compute_observed_summaries(inf_ts, rew_ts, deg_hist)

# Warm up Numba
_ = simulate_fast(0.2, 0.1, 0.3, seed=0)

## 3.1 Run ABC with Different Summary Subsets

We run 50,000 simulations once (shared across all subsets — only the distance
computation changes), then apply different index masks.

In [ ]:
N_SIM = 50_000

# Check if we already have results saved
results_file = '../results/rejection_abc_50k.npz'
if os.path.exists(results_file):
    print("Loading saved results...")
    data = np.load(results_file)
    thetas = data['thetas']
    summaries = data['summaries']
    mads = data['mads']
else:
    print("Running rejection ABC from scratch...")
    t0 = time.time()
    thetas, _, summaries, mads = run_rejection_abc(
        s_obs=s_obs, n_sim=N_SIM, simulator_fn=simulate_fast,
        indices=IDX_ALL, seed=42, verbose=True,
    )
    elapsed = time.time() - t0
    print(f'Completed in {elapsed:.1f}s')
    os.makedirs('../results', exist_ok=True)
    np.savez(results_file, thetas=thetas, summaries=summaries, mads=mads, s_obs=s_obs)

print(f"Loaded {len(thetas)} simulations with {summaries.shape[1]} summary statistics")

In [ ]:
from abc_rejection import normalised_distance

# Define summary subsets to compare
summary_subsets = {
    'Infected only (s1-s5)':     IDX_INFECTED,
    'Rewiring only (s6-s9)':     IDX_REWIRING,
    'Degree only (s10-s12)':     IDX_DEGREE,
    'Infected + Rewiring':       IDX_INFECTED + IDX_REWIRING,
    'Infected + Degree':         IDX_INFECTED + IDX_DEGREE,
    'All 12':                    IDX_ALL,
}

# Recompute distances for each subset and accept at q=1%
q = 0.01
results = {}

for name, indices in summary_subsets.items():
    dists = np.array([
        normalised_distance(summaries[i], s_obs, mads, indices)
        for i in range(len(summaries))
    ])
    acc_t, acc_d, acc_s, thr = accept_quantile(thetas, dists, summaries, quantile=q)
    results[name] = {
        'thetas': acc_t,
        'distances': acc_d,
        'threshold': thr,
        'indices': indices,
    }
    print(f'{name:30s}: {len(acc_t):5d} accepted, threshold={thr:.3f}')

## 3.2 Compare Posterior Marginals Across Summary Subsets

In [ ]:
param_names = [r'$\beta$', r'$\gamma$', r'$\rho$']
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#a65628']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col in range(3):
    ax = axes[0, col]
    for i, (name, res) in enumerate(results.items()):
        ax.hist(res['thetas'][:, col], bins=25, density=True, alpha=0.4,
                color=colors[i], label=name, histtype='stepfilled', linewidth=1.5)
    ax.set_xlabel(param_names[col], fontsize=12)
    ax.set_ylabel('Density')
    ax.set_title(f'Marginal posterior: {param_names[col]}')
    if col == 2:
        ax.legend(fontsize=7, loc='upper left')

# Bottom row: 95% CI widths as bar chart
ci_widths = {name: [] for name in results}
for name, res in results.items():
    for col in range(3):
        lo, hi = np.percentile(res['thetas'][:, col], [2.5, 97.5])
        ci_widths[name].append(hi - lo)

x = np.arange(3)
width = 0.12
for i, (name, widths) in enumerate(ci_widths.items()):
    axes[1, 0].bar(x + i * width, widths, width, color=colors[i], alpha=0.7, label=name)

axes[1, 0].set_xticks(x + 2.5 * width)
axes[1, 0].set_xticklabels(param_names)
axes[1, 0].set_ylabel('95% CI Width')
axes[1, 0].set_title('Posterior Width Comparison')
axes[1, 0].legend(fontsize=6)

# beta-rho scatter: infected-only vs all
for ax_idx, (name, color) in enumerate([('Infected only (s1-s5)', '#e41a1c'), ('All 12', '#a65628')]):
    ax = axes[1, 1 + ax_idx]
    t = results[name]['thetas']
    ax.scatter(t[:, 0], t[:, 2], alpha=0.3, s=8, color=color)
    ax.set_xlabel(r'$\beta$')
    ax.set_ylabel(r'$\rho$')
    ax.set_title(f'{name}')

    # Correlation coefficient
    corr = np.corrcoef(t[:, 0], t[:, 2])[0, 1]
    ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Summary Statistics Comparison (q=1%)', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/summary_stats_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Corner Plots: Infected-Only vs All Summaries

This directly demonstrates the beta-rho confounding when only using infected timeseries statistics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, name in enumerate(['Infected only (s1-s5)', 'All 12']):
    t = results[name]['thetas']
    fig_corner = corner.corner(
        t,
        labels=[r'$\beta$', r'$\gamma$', r'$\rho$'],
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        title_kwargs={'fontsize': 10},
        color=colors[ax_idx * 5],
    )
    fig_corner.suptitle(f'Posterior: {name}', fontsize=13, y=1.02)
    plt.savefig(f'../figures/corner_{name.split()[0].lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3.4 Quantitative Comparison: 95% CI Widths Table

In [ ]:
print(f"{'Summary subset':>30s}  {'beta_width':>10s}  {'gamma_width':>11s}  {'rho_width':>10s}  {'beta_med':>9s}  {'gamma_med':>10s}  {'rho_med':>8s}")
print('-' * 100)

for name, res in results.items():
    t = res['thetas']
    widths = []
    medians = []
    for col in range(3):
        lo, hi = np.percentile(t[:, col], [2.5, 97.5])
        widths.append(hi - lo)
        medians.append(np.median(t[:, col]))
    print(f'{name:>30s}  {widths[0]:>10.4f}  {widths[1]:>11.4f}  {widths[2]:>10.4f}  '
          f'{medians[0]:>9.4f}  {medians[1]:>10.4f}  {medians[2]:>8.4f}')

## 3.5 Pairwise Correlation Analysis

Compute the Pearson correlation between beta and rho in the accepted samples for each summary subset, quantifying the confounding.

In [ ]:
print(f"{'Summary subset':>30s}  {'corr(beta,rho)':>15s}  {'corr(beta,gamma)':>17s}  {'corr(gamma,rho)':>16s}")
print('-' * 85)

for name, res in results.items():
    t = res['thetas']
    r_br = np.corrcoef(t[:, 0], t[:, 2])[0, 1]
    r_bg = np.corrcoef(t[:, 0], t[:, 1])[0, 1]
    r_gr = np.corrcoef(t[:, 1], t[:, 2])[0, 1]
    print(f'{name:>30s}  {r_br:>15.4f}  {r_bg:>17.4f}  {r_gr:>16.4f}')